# hazard_object v1 — full fine-tune on a Colab GPU

Same pipeline as the CPU run in the repo (`tools/build_hazard_dataset.py` → `tools/train_hazard_model.py` → `tools/eval_hazard_model.py`),
but with **all** web images, a bigger teacher (yolo11x), the backbone unfrozen and more epochs.

Runtime → Change runtime type → **T4 GPU** (or better). Total time on a T4: about 2–3 hours.

ข้อควรระวัง: ห้ามเปลี่ยน split หรือเลือก threshold จากชุด test — เลือกจาก val เท่านั้น แล้วค่อยรายงานผลบน test

In [ ]:
# 1. same Ultralytics version as the project
!pip install -q ultralytics==8.4.136
!nvidia-smi --query-gpu=name,memory.total --format=csv
!yolo settings sync=False
import ultralytics, torch
print(ultralytics.__version__, torch.__version__, torch.cuda.is_available())
assert torch.cuda.is_available(), 'select a GPU runtime'

In [ ]:
# 2. code + data (all public, all from GitHub)
%cd /content
!git clone -b dev https://github.com/raftisz/SMART-NURSERY-SAFETY-AI.git repo
# HOD harmful-object dataset (research use only, Ha et al. WACV-W 2024)
!git clone --filter=blob:none --no-checkout --depth 1 https://github.com/poori-nuna/HOD-Benchmark-Dataset.git hod
!cd hod && git sparse-checkout init --no-cone && printf '/dataset/class/knife/\n' > .git/info/sparse-checkout && git checkout HEAD
# Sohas weapon + similar handled objects (DaSCI, CC BY-SA 4.0)
!git clone --filter=blob:none --no-checkout --depth 1 https://github.com/ari-dasci/OD-WeaponDetection.git odw
!cd odw && git sparse-checkout init --no-cone && printf '/Weapons and similar handled objects/Sohas_weapon-Detection-YOLOv5/\n' > .git/info/sparse-checkout && git checkout HEAD
# COCO val2017 (Ultralytics mirror)
!wget -q https://github.com/ultralytics/assets/releases/download/v0.0.0/coco2017val.zip && unzip -q coco2017val.zip -d coco && rm coco2017val.zip

Files below are not on the `dev` branch yet, so the notebook writes them into the clone. Once they are committed, delete these cells.

In [ ]:
%%writefile /content/repo/tools/build_hazard_dataset.py
"""Build the hazard_object fine-tuning dataset (YOLO format, all 80 COCO classes).

Why 80 classes and not just person/scissors/knife
-------------------------------------------------
The new web data has ground truth for `knife` (and `smartphone`) only. If we
trained a 3-class head, Ultralytics would re-initialise the classifier and the
model would have to re-learn `scissors` from the handful of COCO scissors
images we can replay - it would get worse at the primary gate class. Keeping
the COCO head and filling every other class with *teacher pseudo-labels*
(knowledge distillation from a larger COCO model) lets the new knife data
change the knife decision without erasing what the base model knows.

Sources (all public; licences in docs/eval/hazard_v1_dataset.md)
- COCO val2017 (Ultralytics mirror, GT for 80 classes). It is NOT in the
  training set of the pretrained yolo11 weights, so its held-out part is a fair
  test of the baseline. Split A = replay for training, C = validation,
  B = test (never trained on, never used for selection).
- HOD knife (normal + hard cases), Ha et al., WACV-W 2024. GT class 5 = knife.
- Sohas weapon detection (DaSCI, CC BY-SA 4.0): knife held in hand + objects
  handled the same way (smartphone, purse, bill, card) = hard negatives.
  GT knife -> COCO knife (43), GT smartphone -> COCO cell phone (67).
  Pistol-only images are dropped (not a nursery object).

Leakage control
- HOD: near-duplicate images (dHash <= 6 of 64 bits, same rule as
  docs/eval/smoking_cls_dupes.md) are grouped and a group never straddles splits.
- Sohas: the authors' own train/test split is kept; video frames of one clip
  (same filename prefix) stay in one split.

Label rules for non-COCO images (merge_labels)
- GT boxes are kept as-is.
- Teacher boxes of a class that has GT in this source (knife, cell phone) are
  only added when they do not overlap any GT box of that class (IoU < 0.3) AND
  have conf >= complete_conf: the web labels miss knives (checked by eye:
  two knives on a table, one labelled), and an unlabelled knife would be
  taught as background.
- Teacher boxes of every other class are added at conf >= pseudo_conf,
  EXCEPT when they overlap a GT box of another class (IoU >= 0.3): the GT wins.
  Measured on the first build: 363 of 413 teacher "scissors" boxes in the knife
  photos sat on a GT knife - the teacher's own knife/scissors confusion, which
  would otherwise be taught to the student as truth.
- Hard negatives: when a per-source cap is used, at least `neg_fraction` of the
  kept images are images WITHOUT a knife (phones, purses, bills, cards in a hand).
- Test splits keep GT only (no teacher), so the test is not biased toward
  models that behave like the teacher.

Usage
    python tools/build_hazard_dataset.py --coco <coco dir> --hod <HOD repo> \
        --sohas <Sohas_weapon-Detection-YOLOv5 dir> --out <work dir> \
        [--teacher yolo11m.pt] [--max-train-per-source 1200]
"""

from __future__ import annotations

import argparse
import json
import random
import sys
from collections import defaultdict
from dataclasses import dataclass, field
from pathlib import Path

SEED = 0
KNIFE, SCISSORS, PERSON, CELL_PHONE = 43, 76, 0, 67
HOD_KNIFE = 5
SOHAS_TO_COCO = {2: KNIFE, 1: CELL_PHONE}  # 0 pistol, 3 purse, 4 bill, 5 card -> no COCO class
SOHAS_PISTOL = 0
DHASH_BITS = 6
MAX_SIDE = 960  # images are stored resized; training/eval use imgsz <= 960


# --------------------------------------------------------------------------
# pure helpers (unit-tested in tests/test_build_hazard_dataset.py)
# --------------------------------------------------------------------------

Box = tuple[int, float, float, float, float]  # cls, xc, yc, w, h (normalised)


def parse_yolo(text: str) -> list[Box]:
    """YOLO txt -> boxes. Polygon rows (COCO segments) become their bounding box."""
    out: list[Box] = []
    for line in text.splitlines():
        p = line.split()
        if len(p) < 5:
            continue
        c = int(float(p[0]))
        v = [float(x) for x in p[1:]]
        if len(v) == 4:
            out.append((c, *v))
        else:
            xs, ys = v[0::2], v[1::2]
            x1, x2, y1, y2 = min(xs), max(xs), min(ys), max(ys)
            out.append((c, (x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1))
    return out


def format_yolo(boxes: list[Box]) -> str:
    return "".join(f"{c} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n" for c, x, y, w, h in boxes)


def iou_xywh(a: Box, b: Box) -> float:
    ax1, ay1, ax2, ay2 = a[1] - a[3] / 2, a[2] - a[4] / 2, a[1] + a[3] / 2, a[2] + a[4] / 2
    bx1, by1, bx2, by2 = b[1] - b[3] / 2, b[2] - b[4] / 2, b[1] + b[3] / 2, b[2] + b[4] / 2
    iw = max(0.0, min(ax2, bx2) - max(ax1, bx1))
    ih = max(0.0, min(ay2, by2) - max(ay1, by1))
    inter = iw * ih
    union = a[3] * a[4] + b[3] * b[4] - inter
    return inter / union if union > 0 else 0.0


def conflicts_with_gt(box: Box, gt: list[Box], overlap_iou: float = 0.3) -> bool:
    """A teacher box sitting on a GT box of ANOTHER class (e.g. 'scissors' on a GT knife)."""
    return any(g[0] != box[0] and iou_xywh(g, box) >= overlap_iou for g in gt)


def merge_labels(gt: list[Box], teacher: list[tuple[Box, float]], gt_classes: set[int],
                 pseudo_conf: float = 0.4, complete_conf: float = 0.6,
                 overlap_iou: float = 0.3) -> list[Box]:
    """GT + teacher pseudo-labels (see module docstring for the rules)."""
    out = list(gt)
    for box, conf in teacher:
        c = box[0]
        if conflicts_with_gt(box, gt, overlap_iou):
            continue
        if c in gt_classes:
            if conf < complete_conf:
                continue
            if any(g[0] == c and iou_xywh(g, box) >= overlap_iou for g in out):
                continue
        elif conf < pseudo_conf:
            continue
        elif any(g[0] == c and iou_xywh(g, box) >= 0.5 for g in out):
            continue
        out.append(box)
    return out


def hamming(a: int, b: int) -> int:
    return bin(a ^ b).count("1")


def group_near_duplicates(hashes: dict[str, int], max_bits: int = DHASH_BITS) -> dict[str, str]:
    """Union-find over items whose dHash differs by <= max_bits. Returns item -> group root."""
    parent = {k: k for k in hashes}

    def find(k: str) -> str:
        while parent[k] != k:
            parent[k] = parent[parent[k]]
            k = parent[k]
        return k

    keys = sorted(hashes)
    # bucket by 4 x 16-bit bands: two hashes within 6 bits share at least one band? not
    # guaranteed, so compare all pairs; n is a few thousand -> a few million cheap ops.
    vals = [hashes[k] for k in keys]
    for i in range(len(keys)):
        hi = vals[i]
        for j in range(i + 1, len(keys)):
            if hamming(hi, vals[j]) <= max_bits:
                ri, rj = find(keys[i]), find(keys[j])
                if ri != rj:
                    parent[rj] = ri
    return {k: find(k) for k in keys}


def split_groups(groups: dict[str, str], fractions: dict[str, float], seed: int = SEED) -> dict[str, str]:
    """Assign whole groups to splits so each split gets ~fraction of ITEMS. item -> split."""
    members: dict[str, list[str]] = defaultdict(list)
    for item, g in groups.items():
        members[g].append(item)
    order = sorted(members)
    random.Random(seed).shuffle(order)
    total = len(groups)
    names = list(fractions)
    target = {n: fractions[n] * total for n in names}
    filled = {n: 0 for n in names}
    out: dict[str, str] = {}
    for g in order:
        # the split furthest below its target takes the next group
        n = max(names, key=lambda s: (target[s] - filled[s]) / max(target[s], 1e-9))
        for item in members[g]:
            out[item] = n
        filled[n] += len(members[g])
    return out


def sohas_group(name: str) -> str:
    """Clip/source prefix of a Sohas file: 'KravMagaTraining21141.jpg' -> 'KravMagaTraining'."""
    stem = Path(name).stem
    return stem.rstrip("0123456789").rstrip("_") or stem


# --------------------------------------------------------------------------
# I/O
# --------------------------------------------------------------------------

def dhash(path: Path) -> int:
    import cv2
    im = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    im = cv2.resize(im, (9, 8), interpolation=cv2.INTER_AREA)
    bits = (im[:, 1:] > im[:, :-1]).flatten()
    return int("".join("1" if b else "0" for b in bits), 2)


@dataclass
class Item:
    src: Path
    source: str          # coco | hod | sohas
    split: str           # train | val | test
    gt: list[Box]
    gt_classes: set[int] = field(default_factory=set)


def collect_coco(coco_dir: Path, n_train: int, n_val: int) -> list[Item]:
    img_dir, lab_dir = coco_dir / "images" / "val2017", coco_dir / "labels" / "val2017"
    files = sorted(img_dir.glob("*.jpg"))
    rng = random.Random(SEED)
    labels = {f: parse_yolo((lab_dir / (f.stem + ".txt")).read_text())
              if (lab_dir / (f.stem + ".txt")).exists() else [] for f in files}
    hazard = [f for f in files if any(b[0] in (KNIFE, SCISSORS) for b in labels[f])]
    other = [f for f in files if f not in set(hazard)]
    rng.shuffle(hazard)
    rng.shuffle(other)
    # hazard images: 40% train / 10% val / 50% test, so the test keeps half the scissors
    nh = len(hazard)
    h_tr, h_va = hazard[: int(nh * 0.4)], hazard[int(nh * 0.4): int(nh * 0.5)]
    h_te = hazard[int(nh * 0.5):]
    o_tr, o_va, o_te = other[:n_train], other[n_train:n_train + n_val], other[n_train + n_val:]
    items = []
    for split, fs in (("train", h_tr + o_tr), ("val", h_va + o_va), ("test", h_te + o_te)):
        items += [Item(f, "coco", split, labels[f], set(range(80))) for f in fs]
    return items


def collect_hod(hod_dir: Path) -> list[Item]:
    files = sorted((hod_dir / "dataset" / "class" / "knife").glob("*_cases/jpg/*.jpg"))
    print(f"[hod] hashing {len(files)} images for near-duplicate groups ...")
    hashes = {str(f): dhash(f) for f in files}
    groups = group_near_duplicates(hashes)
    split = split_groups(groups, {"train": 0.7, "val": 0.1, "test": 0.2})
    n_groups = len(set(groups.values()))
    print(f"[hod] {len(files)} images in {n_groups} groups")
    items = []
    for f in files:
        lab = f.parent.parent / "txt" / (f.stem + ".txt")
        gt = [(KNIFE, *b[1:]) for b in parse_yolo(lab.read_text()) if b[0] == HOD_KNIFE]
        items.append(Item(f, "hod", split[str(f)], gt, {KNIFE}))
    return items


def collect_sohas(sohas_dir: Path) -> list[Item]:
    base = sohas_dir / "obj_train_data"
    items = []
    train_files = sorted((base / "images" / "train").glob("*"))
    groups = {str(f): sohas_group(f.name) for f in train_files}
    # big sources (knife_, pistol_, ...) are web photos, not clips: group them per file
    sizes: dict[str, int] = defaultdict(int)
    for g in groups.values():
        sizes[g] += 1
    groups = {k: (g if sizes[g] < 300 else k) for k, g in groups.items()}
    tv = split_groups(groups, {"train": 0.9, "val": 0.1})
    for split_dir in ("train", "test"):
        for f in sorted((base / "images" / split_dir).glob("*")):
            lab = base / "labels" / split_dir / (f.stem + ".txt")
            raw = parse_yolo(lab.read_text()) if lab.exists() else []
            if raw and all(b[0] == SOHAS_PISTOL for b in raw):
                continue
            gt = [(SOHAS_TO_COCO[b[0]], *b[1:]) for b in raw if b[0] in SOHAS_TO_COCO]
            split = "test" if split_dir == "test" else tv[str(f)]
            items.append(Item(f, "sohas", split, gt, {KNIFE, CELL_PHONE}))
    return items


def write_image(src: Path, dst: Path) -> None:
    import cv2
    im = cv2.imread(str(src))
    if im is None:
        raise ValueError(f"unreadable image {src}")
    h, w = im.shape[:2]
    s = MAX_SIDE / max(h, w)
    if s < 1:
        im = cv2.resize(im, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA)
    cv2.imwrite(str(dst), im, [cv2.IMWRITE_JPEG_QUALITY, 92])


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--coco", type=Path, required=True, help="dir with images/val2017 and labels/val2017")
    ap.add_argument("--hod", type=Path, required=True, help="HOD-Benchmark-Dataset repo")
    ap.add_argument("--sohas", type=Path, required=True, help="Sohas_weapon-Detection-YOLOv5 dir")
    ap.add_argument("--out", type=Path, required=True)
    ap.add_argument("--teacher", default="yolo11m.pt", help="COCO model for pseudo-labels")
    ap.add_argument("--pseudo-conf", type=float, default=0.4)
    ap.add_argument("--complete-conf", type=float, default=0.6)
    ap.add_argument("--coco-train", type=int, default=1500)
    ap.add_argument("--coco-val", type=int, default=300)
    ap.add_argument("--max-train-per-source", type=int, default=0,
                    help="cap train images per web source (0 = all). For CPU-only runs.")
    ap.add_argument("--max-val-per-source", type=int, default=0)
    ap.add_argument("--neg-fraction", type=float, default=0.3,
                    help="with a cap: share of kept web images that have NO knife (hard negatives)")
    ap.add_argument("--device", default="cpu")
    args = ap.parse_args()

    items = collect_coco(args.coco, args.coco_train, args.coco_val) + collect_hod(args.hod) \
        + collect_sohas(args.sohas)
    rng = random.Random(SEED)
    for cap, split in ((args.max_train_per_source, "train"), (args.max_val_per_source, "val")):
        if not cap:
            continue
        kept = []
        for src in ("hod", "sohas"):
            pool = [i for i in items if i.source == src and i.split == split]
            rng.shuffle(pool)
            pos = [i for i in pool if any(b[0] == KNIFE for b in i.gt)]
            neg = [i for i in pool if not any(b[0] == KNIFE for b in i.gt)]
            n_neg = min(len(neg), round(cap * args.neg_fraction))
            kept += pos[:cap - n_neg] + neg[:n_neg]
        items = [i for i in items if i.source == "coco" or i.split != split] + kept

    out = args.out
    for split in ("train", "val", "test"):
        (out / "images" / split).mkdir(parents=True, exist_ok=True)
        (out / "labels" / split).mkdir(parents=True, exist_ok=True)

    need_teacher = [i for i in items if i.source != "coco" and i.split != "test"]
    teacher_boxes: dict[str, list[tuple[Box, float]]] = {}
    if need_teacher:
        from ultralytics import YOLO
        model = YOLO(args.teacher)
        print(f"[teacher] {args.teacher} on {len(need_teacher)} images ...")
        for k in range(0, len(need_teacher), 16):
            batch = need_teacher[k:k + 16]
            results = model.predict([str(i.src) for i in batch], imgsz=640, conf=min(args.pseudo_conf, 0.25),
                                    device=args.device, verbose=False)
            for it, r in zip(batch, results):
                tb = []
                for xywhn, c, p in zip(r.boxes.xywhn.tolist(), r.boxes.cls.tolist(), r.boxes.conf.tolist()):
                    tb.append(((int(c), *xywhn), float(p)))
                teacher_boxes[str(it.src)] = tb
            if (k // 16) % 20 == 0:
                print(f"[teacher] {k + len(batch)}/{len(need_teacher)}", flush=True)

    manifest = []
    stats: dict[str, dict[str, int]] = defaultdict(lambda: defaultdict(int))
    for n, it in enumerate(items):
        name = f"{it.source}_{n:05d}_{it.src.stem}"[:120]
        labels = it.gt
        if str(it.src) in teacher_boxes:
            labels = merge_labels(it.gt, teacher_boxes[str(it.src)], it.gt_classes,
                                  args.pseudo_conf, args.complete_conf)
        write_image(it.src, out / "images" / it.split / f"{name}.jpg")
        (out / "labels" / it.split / f"{name}.txt").write_text(format_yolo(labels))
        key = f"{it.split}/{it.source}"
        stats[key]["images"] += 1
        stats[key]["knife_boxes"] += sum(b[0] == KNIFE for b in labels)
        stats[key]["scissors_boxes"] += sum(b[0] == SCISSORS for b in labels)
        stats[key]["person_boxes"] += sum(b[0] == PERSON for b in labels)
        stats[key]["added_by_teacher"] += len(labels) - len(it.gt)
        manifest.append({"file": name, "src": str(it.src), "source": it.source, "split": it.split})

    import yaml
    names = coco_names(args.teacher)
    (out / "data.yaml").write_text(yaml.safe_dump(
        {"path": str(out.resolve()), "train": "images/train", "val": "images/val", "test": "images/test",
         "names": names}, sort_keys=False, allow_unicode=True))
    (out / "manifest.json").write_text(json.dumps(manifest, indent=1))
    (out / "stats.json").write_text(json.dumps(stats, indent=2))
    print(json.dumps(stats, indent=2))
    return 0


def coco_names(weights: str) -> dict[int, str]:
    """The 80 COCO class names in Ultralytics order, read from the pretrained weights."""
    from ultralytics import YOLO
    names = dict(YOLO(weights).names)
    if len(names) != 80 or names[KNIFE] != "knife" or names[SCISSORS] != "scissors":
        raise ValueError(f"{weights} is not a COCO-80 model")
    return names


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile /content/repo/tools/train_hazard_model.py
"""Fine-tune the hazard_object model (COCO yolo11 -> + web knife data).

Starts from the pretrained COCO weights and keeps all 80 classes (see
tools/build_hazard_dataset.py for why). Defaults are chosen for a FINE-TUNE,
not training from scratch:
  - lr0 0.002 with SGD (5x below the Ultralytics default): the COCO features
    are already good; a large step erases them (catastrophic forgetting).
  - freeze 10 = the backbone stays as trained on COCO; only neck + head adapt.
    Halves the backward pass on CPU and limits forgetting. Use --freeze 0 on a
    GPU with the full dataset.
  - mosaic on, closed for the last 20% of epochs, fliplr, HSV jitter: standard.
  - scale 0.9: objects far from the camera are small; the stock 0.5 rarely
    shows the model a knife below ~1/4 of its photo size.
  - seed 0, deterministic, so a re-run gives the same weights.

Writes models/hazard/<name>.pt + <name>.train_info.json; refuses to overwrite.

Usage
    python tools/train_hazard_model.py --data <hazard_v1>/data.yaml --epochs 12 --name hazard_v1
"""

from __future__ import annotations

import argparse
import datetime
import hashlib
import json
import shutil
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--data", type=Path, required=True)
    ap.add_argument("--base", default="yolo11n.pt")
    ap.add_argument("--name", default="hazard_v1")
    ap.add_argument("--epochs", type=int, default=12)
    ap.add_argument("--imgsz", type=int, default=640)
    ap.add_argument("--batch", type=int, default=16)
    ap.add_argument("--freeze", type=int, default=10)
    ap.add_argument("--lr0", type=float, default=0.002)
    ap.add_argument("--scale", type=float, default=0.9,
                    help="random zoom range +-scale. 0.9 (vs Ultralytics 0.5) shrinks objects down to 10%% "
                         "of their size, so the model sees many SMALL knives/scissors (far from the camera)")
    ap.add_argument("--device", default="cpu")
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--project", type=Path, default=Path("runs/hazard"))
    ap.add_argument("--fraction", type=float, default=1.0, help="use part of the train split (smoke test)")
    args = ap.parse_args()

    out_dir = ROOT / "models" / "hazard"
    out_pt = out_dir / f"{args.name}.pt"
    if out_pt.exists():
        print(f"{out_pt} exists - pick another --name (models are never overwritten)")
        return 1

    import torch
    import ultralytics
    from ultralytics import YOLO

    train_args = dict(
        data=str(args.data), epochs=args.epochs, imgsz=args.imgsz, batch=args.batch,
        device=args.device, workers=args.workers, seed=0, deterministic=True,
        optimizer="SGD", lr0=args.lr0, lrf=0.1, momentum=0.937, weight_decay=5e-4,
        warmup_epochs=1.0, cos_lr=True, freeze=args.freeze,
        mosaic=1.0, close_mosaic=max(1, round(args.epochs * 0.2)), mixup=0.0,
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, fliplr=0.5, scale=args.scale, translate=0.1,
        patience=max(5, args.epochs // 3), fraction=args.fraction,
        project=str(args.project.resolve()), name=args.name, exist_ok=True, plots=True, verbose=True,
    )
    model = YOLO(args.base)
    model.train(**train_args)

    run = args.project.resolve() / args.name
    best = run / "weights" / "best.pt"
    out_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(best, out_pt)
    digest = hashlib.sha256(out_pt.read_bytes()).hexdigest()
    info = {
        "file": out_pt.name, "sha256": digest,
        "date": datetime.datetime.now().isoformat(timespec="minutes"),
        "base": args.base, "classes": "COCO-80 (unchanged order)",
        "data": str(args.data), "train_args": {k: v for k, v in train_args.items() if k != "project"},
        "ultralytics": ultralytics.__version__, "torch": torch.__version__,
        "device": args.device,
    }
    stats = args.data.parent / "stats.json"
    if stats.exists():
        info["dataset_stats"] = json.loads(stats.read_text())
    (out_dir / f"{args.name}.train_info.json").write_text(json.dumps(info, indent=2, ensure_ascii=False))
    print(json.dumps(info, indent=2, ensure_ascii=False))
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile /content/repo/tools/eval_hazard_model.py
"""Compare hazard_object weights on held-out images, with confidence intervals.

Two kinds of numbers, because they answer different questions:

* box level  - AP50 per class (COCO 101-point, IoU 0.5) and precision/recall at
               a threshold. "Does the model find the object and put the box in
               the right place?"
* image level - what the alert actually depends on. An image "has a hazard"
               when it has a knife or scissors box in the ground truth.
                 hazard recall      = hazard images with >= 1 knife/scissors
                                      detection at >= threshold
                 false-alarm rate   = hazard-FREE images with >= 1 knife/scissors
                                      detection at >= threshold
               (the label does not have to match: a knife reported as scissors
               still raises the same hazard_object alert)

Every number has a 95% bootstrap interval (resampling images, 1000 reps, seed 0).
Differences between two models use a PAIRED bootstrap (same resampled images
for both), so "B is better than A" is only claimed when the interval of the
difference excludes 0.

Thresholds are picked on the VALIDATION split (largest F2 = recall counts
twice as much as precision, because a missed knife is worse than a false
alarm) and then applied unchanged to the TEST split. Never the other way round.

Usage
    python tools/eval_hazard_model.py --data <hazard_v1 dir> \
        --model base=yolo11n.pt@640 --model base960=yolo11n.pt@960 \
        --model v1=models/hazard/hazard_v1.pt@640 --out docs/eval/hazard_v1_eval.md
Predictions are cached next to the dataset (preds_<name>_<split>.json).
"""

from __future__ import annotations

import argparse
import datetime
import hashlib
import json
import random
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np

KNIFE, SCISSORS, PERSON = 43, 76, 0
HAZARD = (KNIFE, SCISSORS)
CLASSES = {PERSON: "person", KNIFE: "knife", SCISSORS: "scissors"}
IOU = 0.5
REPS = 1000
SEED = 0


# --------------------------------------------------------------------------
# metrics (pure, unit-tested)
# --------------------------------------------------------------------------

def iou_xyxy(a, b) -> float:
    iw = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    ih = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = iw * ih
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def match_image(gts: list, preds: list, iou_thr: float = IOU) -> list[tuple[float, bool]]:
    """Greedy matching for ONE class in ONE image. preds: [(conf, box)]. -> [(conf, is_tp)]."""
    used = [False] * len(gts)
    out = []
    for conf, box in sorted(preds, key=lambda p: -p[0]):
        best, best_j = iou_thr, -1
        for j, g in enumerate(gts):
            if used[j]:
                continue
            v = iou_xyxy(box, g)
            if v >= best:
                best, best_j = v, j
        if best_j >= 0:
            used[best_j] = True
        out.append((conf, best_j >= 0))
    return out


def average_precision(scored: list[tuple[float, bool]], n_gt: int) -> float:
    """COCO-style 101-point interpolated AP from (conf, is_tp) over all images."""
    if n_gt == 0:
        return float("nan")
    if not scored:
        return 0.0
    scored = sorted(scored, key=lambda s: -s[0])
    tp = np.cumsum([s[1] for s in scored])
    fp = np.cumsum([not s[1] for s in scored])
    recall = tp / n_gt
    precision = tp / np.maximum(tp + fp, 1e-9)
    # precision envelope
    for i in range(len(precision) - 2, -1, -1):
        precision[i] = max(precision[i], precision[i + 1])
    ap = 0.0
    for r in np.linspace(0, 1, 101):
        idx = np.searchsorted(recall, r, side="left")
        ap += precision[idx] if idx < len(precision) else 0.0
    return ap / 101


#: relative object size = sqrt(box area / image area). In a 1280x720 frame,
#: 0.03 ~ a 29 px object (scissors held 3-4 m away), 0.08 ~ 77 px.
SIZE_BUCKETS = {"small (<3%)": (0.0, 0.03), "medium (3-8%)": (0.03, 0.08), "large (>8%)": (0.08, 9.0)}


class PerImage:
    """Pre-matched results of one model on one image, so bootstrap reps are cheap."""

    def __init__(self, gt: dict[int, list], preds: dict[int, list], wh: tuple[int, int] = (1, 1)):
        self.n_gt = {c: len(gt.get(c, [])) for c in CLASSES}
        area = max(1.0, float(wh[0] * wh[1]))
        # label-agnostic hazard boxes, for recall by object size
        self.hazard_gt = [(b, (((b[2] - b[0]) * (b[3] - b[1])) / area) ** 0.5) for c in HAZARD for b in gt.get(c, [])]
        self.hazard_pred = sorted([p for c in HAZARD for p in preds.get(c, [])], key=lambda p: -p[0])
        self.scored = {c: match_image(gt.get(c, []), preds.get(c, [])) for c in CLASSES}
        self.has_hazard = any(gt.get(c) for c in HAZARD)
        # best hazard confidence in the image (any hazard label)
        self.hazard_conf = max([p[0] for c in HAZARD for p in preds.get(c, [])], default=0.0)


def class_ap(images: list[PerImage], c: int) -> float:
    scored = [s for im in images for s in im.scored[c]]
    return average_precision(scored, sum(im.n_gt[c] for im in images))


def pr_at(images: list[PerImage], c: int, thr: float) -> tuple[float, float]:
    tp = sum(s[1] for im in images for s in im.scored[c] if s[0] >= thr)
    npred = sum(1 for im in images for s in im.scored[c] if s[0] >= thr)
    ngt = sum(im.n_gt[c] for im in images)
    return (tp / npred if npred else float("nan")), (tp / ngt if ngt else float("nan"))


def recall_by_size(images: list[PerImage], thr: float, lo: float, hi: float) -> float:
    """Box recall of knife+scissors (either label) whose relative size is in [lo, hi)."""
    hit = total = 0
    for im in images:
        want = [k for k, (_, size) in enumerate(im.hazard_gt) if lo <= size < hi]
        if not want:
            continue
        total += len(want)
        used = set()
        for conf, box in im.hazard_pred:
            if conf < thr:
                break
            best, best_k = IOU, -1
            for k, (g, _) in enumerate(im.hazard_gt):
                if k in used:
                    continue
                v = iou_xyxy(box, g)
                if v >= best:
                    best, best_k = v, k
            if best_k >= 0:
                used.add(best_k)
        hit += len(used & set(want))
    return hit / total if total else float("nan")


def image_rates(images: list[PerImage], thr: float) -> tuple[float, float]:
    """(hazard recall, false-alarm rate) at image level."""
    pos = [im for im in images if im.has_hazard]
    neg = [im for im in images if not im.has_hazard]
    rec = sum(im.hazard_conf >= thr for im in pos) / len(pos) if pos else float("nan")
    far = sum(im.hazard_conf >= thr for im in neg) / len(neg) if neg else float("nan")
    return rec, far


def best_f2_threshold(images: list[PerImage], c: int, grid=None) -> float:
    grid = grid if grid is not None else [round(x, 2) for x in np.arange(0.05, 0.91, 0.05)]
    best, best_t = -1.0, grid[0]
    for t in grid:
        p, r = pr_at(images, c, t)
        if p != p or r != r or p + r == 0:
            continue
        f2 = 5 * p * r / (4 * p + r)
        if f2 > best:
            best, best_t = f2, t
    return best_t


def bootstrap(fn, images: list[PerImage], reps: int = REPS, seed: int = SEED):
    """(point, lo, hi) of fn(images) with a 95% percentile interval."""
    point = fn(images)
    rng = random.Random(seed)
    n = len(images)
    vals = []
    for _ in range(reps):
        sample = [images[rng.randrange(n)] for _ in range(n)]
        v = fn(sample)
        if v == v:
            vals.append(v)
    if not vals:
        return point, float("nan"), float("nan")
    vals.sort()
    return point, vals[int(0.025 * len(vals))], vals[min(len(vals) - 1, int(0.975 * len(vals)))]


def paired_bootstrap(fn, a: list[PerImage], b: list[PerImage], reps: int = REPS, seed: int = SEED):
    """(point, lo, hi) of fn(b) - fn(a) on the same resampled images."""
    assert len(a) == len(b)
    point = fn(b) - fn(a)
    rng = random.Random(seed)
    n = len(a)
    vals = []
    for _ in range(reps):
        idx = [rng.randrange(n) for _ in range(n)]
        v = fn([b[i] for i in idx]) - fn([a[i] for i in idx])
        if v == v:
            vals.append(v)
    vals.sort()
    if not vals:
        return point, float("nan"), float("nan")
    return point, vals[int(0.025 * len(vals))], vals[min(len(vals) - 1, int(0.975 * len(vals)))]


# --------------------------------------------------------------------------
# I/O
# --------------------------------------------------------------------------

def load_gt(label_path: Path, w: int, h: int) -> dict[int, list]:
    gt: dict[int, list] = defaultdict(list)
    if label_path.exists():
        for line in label_path.read_text().splitlines():
            p = line.split()
            if len(p) < 5:
                continue
            c = int(p[0])
            if c not in CLASSES:
                continue
            x, y, bw, bh = map(float, p[1:5])
            gt[c].append(((x - bw / 2) * w, (y - bh / 2) * h, (x + bw / 2) * w, (y + bh / 2) * h))
    return gt


def predict_cached(weights: str, imgsz: int, images: list[Path], cache: Path, device: str,
                   zoom: bool = False) -> dict:
    """conf 0.01 so every threshold can be evaluated afterwards. zoom=True adds the
    person-guided zoom pass of src/detectors/hazard_zoom.py (default ZoomConfig)."""
    if cache.exists():
        return json.loads(cache.read_text())
    from ultralytics import YOLO
    model = YOLO(weights)
    names = {v: k for k, v in model.names.items()}
    want = {names[n]: c for c, n in CLASSES.items() if n in names}
    hazard_ids = sorted(i for i, c in want.items() if c in HAZARD)
    if zoom:
        sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
        from src.detectors.hazard_zoom import ZoomConfig, zoom_detect
        zcfg = ZoomConfig(enabled=True)

        def predict_crops(crops):
            rs = model.predict(crops, imgsz=zcfg.crop_imgsz, conf=0.01, classes=hazard_ids,
                               device=device, verbose=False)
            return [[(str(want[int(c)]), float(p), tuple(b))
                     for b, c, p in zip(r.boxes.xyxy.tolist(), r.boxes.cls.tolist(), r.boxes.conf.tolist())]
                    for r in rs]
    out = {}
    for k in range(0, len(images), 16):
        batch = images[k:k + 16]
        for path, r in zip(batch, model.predict([str(p) for p in batch], imgsz=imgsz, conf=0.01,
                                                classes=sorted(want), device=device, verbose=False)):
            h, w = r.orig_shape
            boxes = [[want[int(c)], float(p), *map(float, b)]
                     for b, c, p in zip(r.boxes.xyxy.tolist(), r.boxes.cls.tolist(), r.boxes.conf.tolist())]
            if zoom:
                full = [(str(c), p, tuple(b)) for c, p, *b in boxes]
                full_for_zoom = [("person" if c == str(PERSON) else c, p, b) for c, p, b in full]
                merged, _ = zoom_detect(r.orig_img, full_for_zoom, "person", zcfg, predict_crops)
                boxes = [[PERSON if c == "person" else int(c), p, *b] for c, p, b in merged]
            out[path.name] = {"wh": [w, h], "boxes": boxes}
        if (k // 16) % 25 == 0:
            print(f"  [{cache.stem}] {k + len(batch)}/{len(images)}", flush=True)
    cache.write_text(json.dumps(out))
    return out


def per_image(data: Path, split: str, preds: dict, files: list[Path]) -> list[PerImage]:
    res = []
    for f in files:
        rec = preds[f.name]
        w, h = rec["wh"]
        gt = load_gt(data / "labels" / split / (f.stem + ".txt"), w, h)
        pr: dict[int, list] = defaultdict(list)
        for c, conf, *box in rec["boxes"]:
            pr[int(c)].append((conf, box))
        res.append(PerImage(gt, pr, (w, h)))
    return res


def sha(path: str) -> str:
    p = Path(path)
    return hashlib.sha256(p.read_bytes()).hexdigest()[:16] if p.is_file() else "(stock download)"


def fmt(v, pct=True) -> str:
    point, lo, hi = v
    if point != point:
        return "n/a"
    if pct:
        return f"{100 * point:.1f} [{100 * lo:.1f}, {100 * hi:.1f}]"
    return f"{point:.3f} [{lo:.3f}, {hi:.3f}]"


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--data", type=Path, required=True)
    ap.add_argument("--model", action="append", required=True, help="name=weights@imgsz")
    ap.add_argument("--baseline", help="name of the model differences are measured against (default: first)")
    ap.add_argument("--out", type=Path)
    ap.add_argument("--device", default="cpu")
    ap.add_argument("--fixed-threshold", type=float, default=0.25,
                    help="the threshold config/core.yaml uses today; reported next to the tuned one")
    ap.add_argument("--max-coco-negatives", type=int, default=1000,
                    help="cap on hazard-free COCO test images (CPU time)")
    args = ap.parse_args()

    models = []
    for spec in args.model:
        name, rest = spec.split("=", 1)
        weights, size = rest.rsplit("@", 1)
        zoom = size.endswith("+zoom")
        models.append((name, weights, int(size.replace("+zoom", "")), zoom))
    base_name = args.baseline or models[0][0]

    manifest = {m["file"]: m for m in json.loads((args.data / "manifest.json").read_text())}
    files: dict[str, list[Path]] = {}
    for split in ("val", "test"):
        fs = sorted((args.data / "images" / split).glob("*.jpg"))
        if split == "test":
            rng = random.Random(SEED)
            coco_neg = [f for f in fs if manifest[f.stem]["source"] == "coco"
                        and not any(c in HAZARD for c in load_gt(args.data / "labels" / split / (f.stem + ".txt"), 1, 1))]
            rng.shuffle(coco_neg)
            drop = set(coco_neg[args.max_coco_negatives:])
            fs = [f for f in fs if f not in drop]
        files[split] = fs

    results: dict[str, dict[str, list[PerImage]]] = {}
    for name, weights, imgsz, zoom in models:
        results[name] = {}
        for split in ("val", "test"):
            cache = args.data / f"preds_{name}_{split}.json"
            preds = predict_cached(weights, imgsz, files[split], cache, args.device, zoom)
            results[name][split] = per_image(args.data, split, preds, files[split])

    def subset(name: str, split: str, source: str | None) -> list[PerImage]:
        ims = results[name][split]
        if source is None:
            return ims
        return [im for im, f in zip(ims, files[split]) if manifest[f.stem]["source"] == source]

    lines = [f"# hazard_object: model comparison on held-out images",
             "", f"- date: {datetime.datetime.now():%Y-%m-%d %H:%M}",
             f"- dataset: `{args.data.name}` (built by tools/build_hazard_dataset.py; split by source, "
             "near-duplicates grouped)",
             f"- test images: {len(files['test'])} (COCO hazard-free images capped at {args.max_coco_negatives})",
             "- 95% bootstrap intervals in brackets (images resampled, 1000 reps)",
             "- **These are web/COCO photos, not the nursery camera.** They rank models; they do not "
             "replace the QA clips (tools/eval_clips.py) or the Phase 2 Decision Gate.", "",
             "| model | weights | sha256 | imgsz | person-guided zoom |", "|---|---|---|---|---|"]
    for name, weights, imgsz, zoom in models:
        lines.append(f"| {name} | `{weights}` | `{sha(weights)}` | {imgsz} | {'yes' if zoom else 'no'} |")

    thresholds: dict[str, dict[int, float]] = {}
    for name, *_ in models:
        val = results[name]["val"]
        thresholds[name] = {c: best_f2_threshold(val, c) for c in HAZARD}
        # image-level threshold: the lower of the two (an alert needs either label)
    lines += ["", "## 1. Box level, test split (AP50 %)", "",
              "| model | source | person | knife | scissors |", "|---|---|---|---|---|"]
    for name, *_ in models:
        for source in ("coco", "hod", "sohas"):
            ims = subset(name, "test", source)
            row = []
            for c in (PERSON, KNIFE, SCISSORS):
                if sum(im.n_gt[c] for im in ims) == 0:
                    row.append("-")
                else:
                    row.append(fmt(bootstrap(lambda s, c=c: class_ap(s, c), ims)))
            lines.append(f"| {name} | {source} | " + " | ".join(row) + " |")

    lines += ["", "## 2. Image level (what raises the alert), test split", "",
              f"Threshold `tuned` = min of the per-class F2-best thresholds on the validation split; "
              f"`fixed` = {args.fixed_threshold} (today's config).", "",
              "| model | threshold | source | hazard recall % | false-alarm rate % |", "|---|---|---|---|---|"]
    for name, *_ in models:
        tuned = min(thresholds[name].values())
        for label, thr in (("fixed", args.fixed_threshold), ("tuned", tuned)):
            for source in (None, "coco", "hod", "sohas"):
                ims = subset(name, "test", source)
                rec = bootstrap(lambda s, t=thr: image_rates(s, t)[0], ims)
                far = bootstrap(lambda s, t=thr: image_rates(s, t)[1], ims)
                lines.append(f"| {name} | {label} {thr:.2f} | {source or 'all'} | {fmt(rec)} | {fmt(far)} |")

    lines += ["", f"## 2b. Recall by object size (far objects are small), test split, threshold {args.fixed_threshold}", "",
              "Box recall of knife + scissors (either label counts). Size = sqrt(box area / image area).", "",
              "| model | " + " | ".join(SIZE_BUCKETS) + " |", "|---|" + "---|" * len(SIZE_BUCKETS)]
    for name, *_ in models:
        ims = results[name]["test"]
        cells = [fmt(bootstrap(lambda s, lo=lo, hi=hi: recall_by_size(s, args.fixed_threshold, lo, hi), ims))
                 for lo, hi in SIZE_BUCKETS.values()]
        lines.append(f"| {name} | " + " | ".join(cells) + " |")
    counts = [sum(1 for im in results[models[0][0]]["test"] for _, sz in im.hazard_gt if lo <= sz < hi)
              for lo, hi in SIZE_BUCKETS.values()]
    lines.append("| (objects) | " + " | ".join(str(c) for c in counts) + " |")

    lines += ["", "## 3. Per-class thresholds chosen on validation (F2)", "",
              "| model | knife | scissors |", "|---|---|---|"]
    for name, *_ in models:
        lines.append(f"| {name} | {thresholds[name][KNIFE]:.2f} | {thresholds[name][SCISSORS]:.2f} |")

    lines += ["", f"## 4. Paired differences vs `{base_name}` (test split, points; + = better)", "",
              "| model | knife AP50 (all) | scissors AP50 (coco) | person AP50 (coco) | "
              f"hazard recall @ {args.fixed_threshold} | small-object recall @ {args.fixed_threshold} | "
              f"false-alarm rate @ {args.fixed_threshold} (− = better) |",
              "|---|---|---|---|---|---|---|"]
    for name, *_ in models:
        if name == base_name:
            continue
        a_all, b_all = results[base_name]["test"], results[name]["test"]
        a_coco, b_coco = subset(base_name, "test", "coco"), subset(name, "test", "coco")
        t = args.fixed_threshold
        cells = [
            fmt(paired_bootstrap(lambda s: class_ap(s, KNIFE), a_all, b_all)),
            fmt(paired_bootstrap(lambda s: class_ap(s, SCISSORS), a_coco, b_coco)),
            fmt(paired_bootstrap(lambda s: class_ap(s, PERSON), a_coco, b_coco)),
            fmt(paired_bootstrap(lambda s: image_rates(s, t)[0], a_all, b_all)),
            fmt(paired_bootstrap(lambda s: recall_by_size(s, t, *SIZE_BUCKETS["small (<3%)"]), a_all, b_all)),
            fmt(paired_bootstrap(lambda s: image_rates(s, t)[1], a_all, b_all)),
        ]
        lines.append(f"| {name} | " + " | ".join(cells) + " |")

    n_sc = sum(im.n_gt[SCISSORS] for im in subset(models[0][0], "test", "coco"))
    lines += ["", "## Notes", "",
              f"- scissors test boxes: {n_sc} (COCO only). Small n: read the interval, not the point.",
              "- Web knife labels are incomplete (checked by eye); test labels were NOT completed by a model, "
              "so a correct box on an unlabelled knife counts as a false positive for every model alike.",
              "- Sohas hard negatives (smartphone, purse, bill, card held in the hand) are in the "
              "false-alarm rate of the `sohas` rows."]
    text = "\n".join(lines) + "\n"
    print(text)
    if args.out:
        args.out.write_text(text)
        (args.out.with_suffix(".json")).write_text(json.dumps(
            {n: {str(k): v for k, v in t.items()} for n, t in thresholds.items()}, indent=2))
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile /content/repo/src/detectors/hazard_zoom.py
"""Person-guided zoom: a second, close-up look around small (far-away) people.

Why: a pair of scissors held by a child 3-4 m from the camera is ~10-20 px long
in a 1280x720 frame. After the letterbox resize to imgsz the detector sees even
fewer pixels, and the object blends into the background - this is the
"detects near, misses far" failure. People, on the other hand, are big and are
detected reliably at that distance. So: take the person boxes the same model
already produced, cut out each SMALL person (plus a margin for outstretched
arms), and run the detector again on that crop. The crop is resized up to
crop_imgsz, so the object becomes 3-6x larger to the network, and most of the
distracting background is gone.

Cost: one extra inference per crop (batched), only for far-away people, and
only every `every_n` frames if configured. Near people are skipped: they are
already large in the full frame.

This module is pure geometry + merging; the model call is passed in as a
function so it is unit-tested without a model (tests/test_hazard_zoom.py).
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Callable, Sequence

Box = tuple[float, float, float, float]          # x1, y1, x2, y2 in pixels
Det = tuple[str, float, Box]                      # label, confidence, box


@dataclass
class ZoomConfig:
    enabled: bool = False
    max_crops: int = 3             # at most this many people per frame
    max_person_height: float = 0.6  # only people shorter than this fraction of the frame (far away)
    min_person_conf: float = 0.3
    pad_x: float = 0.6             # margin as a fraction of person width (arms reach sideways)
    pad_y: float = 0.15
    min_crop: int = 160            # px; tiny crops are grown to at least this
    crop_imgsz: int = 640
    every_n: int = 1               # run the zoom pass on every n-th frame the detector runs
    merge_iou: float = 0.5

    @classmethod
    def from_dict(cls, data: dict | None) -> "ZoomConfig":
        data = dict(data or {})
        unknown = set(data) - set(cls.__dataclass_fields__)
        if unknown:
            raise ValueError(f"unknown zoom option(s): {sorted(unknown)}")
        return cls(**data)


def iou(a: Box, b: Box) -> float:
    iw = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    ih = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = iw * ih
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def crop_boxes(persons: Sequence[Det], width: int, height: int, cfg: ZoomConfig) -> list[tuple[int, int, int, int]]:
    """Integer crop windows around the small, confident people (largest crops not needed)."""
    small = [p for p in persons
             if p[1] >= cfg.min_person_conf and (p[2][3] - p[2][1]) < cfg.max_person_height * height]
    small.sort(key=lambda p: -p[1])
    crops: list[tuple[int, int, int, int]] = []
    for _, _, (x1, y1, x2, y2) in small:
        w, h = x2 - x1, y2 - y1
        cx1, cy1 = x1 - cfg.pad_x * w, y1 - cfg.pad_y * h
        cx2, cy2 = x2 + cfg.pad_x * w, y2 + cfg.pad_y * h
        # grow to the minimum size around the centre
        cw, ch = cx2 - cx1, cy2 - cy1
        if cw < cfg.min_crop:
            cx1 -= (cfg.min_crop - cw) / 2
            cx2 += (cfg.min_crop - cw) / 2
        if ch < cfg.min_crop:
            cy1 -= (cfg.min_crop - ch) / 2
            cy2 += (cfg.min_crop - ch) / 2
        box = (int(max(0, cx1)), int(max(0, cy1)), int(min(width, cx2)), int(min(height, cy2)))
        if box[2] - box[0] < 8 or box[3] - box[1] < 8:
            continue
        # a crop mostly inside an already chosen crop adds nothing
        if any(iou(box, c) > 0.6 for c in crops):
            continue
        crops.append(box)
        if len(crops) >= cfg.max_crops:
            break
    return crops


def merge(full: list[Det], zoomed: list[Det], merge_iou: float) -> list[Det]:
    """Full-frame detections + zoom detections; a same-label pair overlapping by
    >= merge_iou keeps only the more confident box."""
    out = list(full)
    for z in sorted(zoomed, key=lambda d: -d[1]):
        clash = [i for i, f in enumerate(out) if f[0] == z[0] and iou(f[2], z[2]) >= merge_iou]
        if not clash:
            out.append(z)
            continue
        i = max(clash, key=lambda k: out[k][1])
        if z[1] > out[i][1]:
            out[i] = z
    return out


def zoom_detect(image, full: list[Det], person_label: str, cfg: ZoomConfig,
                predict: Callable[[list], list[list[Det]]]) -> tuple[list[Det], int]:
    """Run `predict` on the person crops of `image` and merge. Returns (detections, n_crops).

    `predict(crops)` gets a list of image arrays and returns, per crop, detections
    in CROP pixel coordinates (only the labels of interest)."""
    h, w = image.shape[:2]
    persons = [d for d in full if d[0].lower() == person_label]
    windows = crop_boxes(persons, w, h, cfg)
    if not windows:
        return full, 0
    crops = [image[y1:y2, x1:x2] for x1, y1, x2, y2 in windows]
    zoomed: list[Det] = []
    for (x1, y1, _, _), dets in zip(windows, predict(crops)):
        for label, conf, (a, b, c, d) in dets:
            zoomed.append((label, conf, (a + x1, b + y1, c + x1, d + y1)))
    return merge(full, zoomed, cfg.merge_iou), len(windows)


In [ ]:
# 3. build the dataset: no caps, yolo11x teacher on the GPU
%cd /content/repo
!python tools/build_hazard_dataset.py --coco /content/coco/coco --hod /content/hod \
    --sohas "/content/odw/Weapons and similar handled objects/Sohas_weapon-Detection-YOLOv5" \
    --out /content/hazard_full --teacher yolo11x.pt --coco-train 2000 --coco-val 300 --device 0

In [ ]:
# 4. fine-tune: backbone unfrozen, imgsz 960 (the size config/core.yaml runs at)
!python tools/train_hazard_model.py --data /content/hazard_full/data.yaml --name hazard_v1_full \
    --epochs 40 --imgsz 960 --batch 16 --freeze 0 --lr0 0.002 --device 0 --workers 4

In [ ]:
# 5. before -> after on the held-out test split (thresholds picked on val), with and without person-guided zoom
!python tools/eval_hazard_model.py --data /content/hazard_full --device 0 \
    --model base=yolo11n.pt@960 --model base_zoom=yolo11n.pt@960+zoom \
    --model v1_full=models/hazard/hazard_v1_full.pt@960 --model v1_full_zoom=models/hazard/hazard_v1_full.pt@960+zoom \
    --out docs/eval/hazard_v1_full_eval.md

In [ ]:
# 6. pack the result: download hazard_v1_full.zip and unzip it at the repo root on the Mac
!cd /content/repo && zip -q -r /content/hazard_v1_full.zip models/hazard docs/eval/hazard_v1_full_eval.md docs/eval/hazard_v1_full_eval.json
!ls -la /content/hazard_v1_full.zip
!cat /content/repo/docs/eval/hazard_v1_full_eval.md
from google.colab import files
files.download('/content/hazard_v1_full.zip')